In [1]:
%config Completer.use_jedi = True

In [2]:
#!pip uninstall -y peft transformers trl accelerate diffusers bitsandbytes sentence-transformers

In [3]:
!pip install -U transformers accelerate bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 19.8 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0


In [4]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594

In [5]:
!pip install huggingface_hub[hf_xet]
#!pip install --upgrade accelerate peft trl bitsandbytes transformers
#!pip uninstall -y peft trl transformers accelerate bitsandbytes

In [6]:
!pip install evaluate bert-score sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.5 MB/s eta 0:00:00


In [7]:
#!pip install --no-cache-dir transformers==4.41.2

In [8]:
#!pip install --no-cache-dir peft==0.11.1


In [9]:
#!pip install --no-cache-dir trl==0.9.6


In [10]:
#!pip install --no-cache-dir accelerate==0.30.1

In [11]:
#!pip install --no-cache-dir diffusers==0.29.2

In [12]:
#!pip install --no-cache-dir bitsandbytes

In [13]:
import torch
from huggingface_hub import login
#from google.colab import userdata
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTTrainer, SFTConfig
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm
import evaluate
from sacrebleu import sentence_bleu, corpus_bleu
import pandas as pd

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [14]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

In [15]:
ds = load_dataset("Ahmad0067/MedSynth", split='train')

README.md: 0.00B [00:00, ?B/s]

MedSynth_huggingface_final.csv:   0%|          | 0.00/78.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10240 [00:00<?, ? examples/s]

In [16]:
split1 = ds.train_test_split(test_size=0.2, seed=42)

train_ds = split1["train"]
test_ds = split1["test"]

In [17]:
train_ds

Dataset({
    features: [' Note', 'Dialogue', 'ICD10', 'ICD10_desc'],
    num_rows: 8192
})

In [18]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

#Change QLoRA to use Unsloth
#tokenizer = AutoTokenizer.from_pretrained(model_name)

#bnb_config = BitsAndBytesConfig(
#    load_in_4bit=True,
#    bnb_4bit_quant_type="nf4",
#    bnb_4bit_compute_dtype=torch.float16,
#    bnb_4bit_use_double_quant=True
#)

#Change QLoRA to use Unsloth
#model = AutoModelForCausalLM.from_pretrained(
#    model_name,
#    quantization_config=bnb_config,
#    device_map="auto"
#)
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit = True,
)


# LoRA configuration
#lora_config = LoraConfig(
#    r=16,
#    lora_alpha=32,
#    target_modules=["q_proj","v_proj"],
#    lora_dropout=0.05,
#    bias="none",
#    task_type="CAUSAL_LM"
#)
#model = get_peft_model(model, lora_config)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# Format training text
def format_prompt(example):

    prompt = f"""
### Instruction:
Convert this Dialogue to SOAP Format Note. Do not add information that is not in the Dialogue.

### Dialogue:
{example['Dialogue']}

### Response:
{example[' Note']}
"""

    #tokenized = tokenizer(
    #    prompt,
    #    truncation=True,
    #    max_length=max_seq_length,
    #    padding="max_length",
    #)

    #tokenized["labels"] = tokenized["input_ids"].copy()
    #return tokenized
    return {"text": prompt}

dataset = train_ds.map(format_prompt, remove_columns=train_ds.column_names)

#for i, example in enumerate(dataset):
#    if not isinstance(example["labels"], list) or len(example["labels"]) == 0:
#        raise ValueError(f"Bad example at index {i}: {example}")

#Change QLoRA to use Unsloth
# Training arguments
#training_args = SFTConfig(
#    output_dir="./llama_soap_model",
#    per_device_train_batch_size=4,
#    gradient_accumulation_steps=4,
#    learning_rate=2e-4,
#    num_train_epochs=3,
#    logging_steps=10,
#    save_steps=200,
#    fp16=False,
#    bf16=False,
#    packing=True,
#    optim="paged_adamw_8bit"
#)

# Trainer
#trainer = SFTTrainer(
#    model=model,
#    train_dataset=dataset,
#    processing_class=tokenizer,
#    args=training_args,
#)
training_args = TrainingArguments(
    output_dir="./llama_soap_model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    optim="adamw_8bit",
    average_tokens_across_devices=False
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = training_args,
    packing = True
)


trainer.train()

trainer.save_model("llama_soap_extractor")
tokenizer.save_pretrained("llama_soap_extractor")

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/8192 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/8192 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,192 | Num Epochs = 3 | Total steps = 1,536
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 3,407,872 of 1,239,222,272 (0.28% trained)


Step,Training Loss
50,1.208768
100,1.036774
150,0.988023
200,0.969633
250,0.952140
300,0.931636
350,0.924916
400,0.914440
450,0.905949
500,0.905693


('llama_soap_extractor/tokenizer_config.json',
 'llama_soap_extractor/chat_template.jinja',
 'llama_soap_extractor/tokenizer.json')

In [19]:
base_model = "meta-llama/Llama-3.2-1B-Instruct"
adapter = "llama_soap_extractor"

#Change from QLoRA to us Unsloth
#tokenizer = AutoTokenizer.from_pretrained(base_model)

#model = AutoModelForCausalLM.from_pretrained(
#    base_model,
#    device_map="auto"
#)

#model = PeftModel.from_pretrained(model, adapter)

model, tokenizer = FastLanguageModel.from_pretrained(
    "llama_soap_extractor",
    dtype=torch.float16
)

model = FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


prompt = """
### Instruction:
Convert this Dialogue to SOAP Format Note.

### Dialogue:
Doctor: Your blood pressure is 128/82 mmHg, heart rate 72 bpm, temperature 98.6°F.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.2
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [20]:
def build_prompt(dialogue):

    return f"""
### Instruction:
Convert this Dialogue to SOAP Format Note. Do not add information that is not in the Dialogue.

### Dialogue:
{dialogue}

### Response:
"""

In [21]:
def generate_batch(dialogues, batch_size=8):

    predictions = []

    for i in tqdm(range(0, len(dialogues), batch_size)):

        batch_dialogues = dialogues[i:i+batch_size]

        prompts = [build_prompt(d) for d in batch_dialogues]

        inputs = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=1000,
                temperature=0.2,
                top_p=0.9
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        preds = [d.split("### Response:")[-1].strip() for d in decoded]

        predictions.extend(preds)

    return predictions

In [22]:
dialogues = test_ds["Dialogue"]
references = test_ds[" Note"]

predictions = generate_batch(dialogues, batch_size=8)

  0%|          | 0/256 [00:00<?, ?it/s]--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/trait

In [23]:
rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references
)

print(rouge_scores)

{'rouge1': np.float64(0.50264956448577), 'rouge2': np.float64(0.2863697986080046), 'rougeL': np.float64(0.3430324692642791), 'rougeLsum': np.float64(0.4819604933589833)}


In [24]:
bertscore = evaluate.load("bertscore")

bert_scores = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en"
)

print("BERTScore F1:", sum(bert_scores["f1"]) / len(bert_scores["f1"]))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1: 0.9051646308216732


In [25]:
#bleu = evaluate.load("bleu")

#bleu_scores = bleu.compute(
#    predictions=predictions, 
#    references=[[r] for r in references]
#)

#print("BLEU:", bleu_scores)
corpus_score = corpus_bleu(predictions, references)

print("Corpus BLEU:", corpus_score)

Corpus BLEU: BLEU = 0.00 1.5/0.0/0.0/0.0 (BP = 1.000 ratio = 908.084 hyp_len = 1099690 ref_len = 1211)


In [26]:
meteor = evaluate.load("meteor")

result = meteor.compute(predictions=predictions, references=references)
print("METEOR:", result)

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


METEOR: {'meteor': np.float64(0.5091806492656165)}


In [27]:
df = pd.DataFrame({
    "Dialogue": dialogues,
    "Reference": references,
    "Prediction": predictions
})

df.to_csv("soap_model_predictions.csv", index=False)